
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>



# Lab - Building and Registering a Retrieval Agent

## Overview

In this lab, you will build and register a production-ready retrieval agent using Databricks Mosaic AI. You will create a LangChain-based agent that combines large language models with a vector search index, implement MLflow tracing for observability, and register the agent to Unity Catalog's Model Registry with proper versioning and aliases.

## Learning Objectives

By the end of this lab, you will be able to:

1. Enable MLflow tracing for LangChain to monitor agent behavior.
1. Build a retrieval agent using LangChain with vector search integration.
1. Analyze agent execution traces to identify performance characteristics.
1. Write agent code to a Python file following the "agent as code" pattern.
1. Register the agent model to Unity Catalog with an alias.
1. Test the registered model to validate functionality.

## Requirements

- A pre-created **vector search endpoint**. This is pre-created for you.
- **Serverless Compute (environment version 5)**. Follow the instructions [here](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) to select the appropriate environment version.


**📌 Your Task: In this lab, your task is to replace `<FILL_IN>` sections with appropriate code.**

## Setup

Run the code below to install required libraries and configure your classroom environment. This step ensures all dependencies are available and your workspace is ready for the demo.

In [0]:
%run ../Includes/Classroom-Setup-04 $section="lab"

## A. Building a Retrieval Agent with LangChain

In this section, you will build a retrieval agent using **LangChain**. The agent will use Unity Catalog's vector search as a tool, allowing it to dynamically retrieve relevant context when answering user questions.

You will follow an **"agent as code"** approach by writing your agent implementation to a Python file (`agent.py`). This is the recommended method when logging models with MLflow.

**Dataset Information:** The vector search index contains data from a fictitious robot manufacturer for a robot named Orion. Documents include context-grounded answers sourced from internal design manuals, compliance documentation, and maintenance guides.

### A1. Task 1 - Enable MLflow Tracing

Before building the agent, you need to **enable MLflow tracing for LangChain** so you can observe agent inputs, tool usage, and outputs in detail.

**Your Task:**

1. Enable MLflow autologging for LangChain using the appropriate method.

In [0]:
# Import mlflow and enable autologging for LangChain

<FILL_IN>

In [0]:
%skip
import mlflow
mlflow.langchain.autolog()

### A2. Task 2 - Create a LangChain Agent

Now you will create a LangChain agent that uses a vector search retriever tool to access the Orion knowledge base.

**Your Task:**

1. Define the LLM endpoint name as `"databricks-gpt-oss-20b"`.
1. Complete the `build_agent` function by:
   * Creating a `ChatDatabricks` model with the provided endpoint and `max_tokens=300`.
   * Creating a `VectorSearchRetrieverTool` with:
     - `name="orion_knowledge_search_lab"`
     - `index_name` from the parameter
     - `description="Search Orion knowledge base for relevant information"`
     - `num_results` from the parameter (5 results)
   * Using the provided system prompt.
   * Creating an agent using `create_agent` with the model, tools list, system prompt, and checkpointer.
1. Test the agent with the question "What is Orion?"

In [0]:
from langchain.agents import create_agent
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool

llm_endpoint_name = <FILL_IN>

def build_agent(llm_endpoint:str, index_name: str, num_results: int = 3):
    model = <FILL_IN>

    vs_tool = <FILL_IN>

    # Optional: use an in memory saver to save the agent's state
    checkpointer = <FILL_IN>

    system_prompt = """You are the Orion Knowledge Assistant (OKA). Respond in a clear, professional, and factual tone appropriate for engineers and technical staff. Use only verified information from Orion's internal documents, and include source references when available. If the answer cannot be found, clearly state that and suggest related sections or next steps. Do not speculate, make assumptions, or provide information outside the provided context."""

    agent = <FILL_IN>
    return agent

# Quick smoke test
agent = build_agent(llm_endpoint_name, index_name, 3)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is Orion?"}]}
)
print(response['messages'][-1].content)

In [0]:
%skip
from langchain.agents import create_agent
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool

llm_endpoint_name = "databricks-gpt-oss-20b"

def build_agent(llm_endpoint:str, index_name: str, num_results: int = 5):
    model = ChatDatabricks(
        endpoint=llm_endpoint,
        max_tokens=300,
    )

    vs_tool = VectorSearchRetrieverTool(
        name="orion_knowledge_search_lab",
        index_name=index_name,
        description="Search Orion knowledge base for relevant information",
        num_results=num_results,
    )

    system_prompt = """You are the Orion Knowledge Assistant (OKA). Respond in a clear, professional, and factual tone appropriate for engineers and technical staff. Use only verified information from Orion's internal documents, and include source references when available. If the answer cannot be found, clearly state that and suggest related sections or next steps. Do not speculate, make assumptions, or provide information outside the provided context."""

    agent = create_agent(
        model=model, 
        tools=[vs_tool], 
        system_prompt=system_prompt,
    )
    return agent

# Quick smoke test
agent = build_agent(llm_endpoint_name, vs_index_name, 3)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is Orion?"}]}
)
print(response['messages'][-1].content)

### A3. Task 3 - Review MLflow Tracing UI

The MLflow Tracing UI provides a comprehensive view of your agent's execution and tool usage. The output above shows the tracing UI.

**Your Task:**

1. Find the name of the longest step in the execution timeline.
1. Identify which tool or model was responsible for that step.
1. Check the total number of tokens used in the agent run (input, output, and total tokens).


## B. Logging and Registering the Agent to Model Registry

In this section, you will prepare the agent for production by writing it to a Python file and registering it to Unity Catalog's Model Registry. You will use the configuration file from the demo notebook instead of creating a new one.

### B1. Write Agent Code and Configuration Files

The following steps create the required files for logging and registering your agent:

- **`agent-lab.py`**: This file wraps the agent logic using MLflow `pyfunc`, enabling support for `ResponseAgent` requests.
- **`agent-config-lab.yaml`**: This file contains the agent configuration.

No action is needed in this section. Simply run the code to generate the necessary files.


In [0]:
%%writefile agent-lab.py
import os
from typing import Any, Dict, List

import yaml
import mlflow
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import ResponsesAgentRequest, ResponsesAgentResponse

from uuid import uuid4

from langchain.agents import create_agent
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool

# Load agent configuration from YAML file
def _load_config(path: str = "agent-config-lab.yaml") -> Dict[str, Any]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Config file not found at '{path}'")
    with open(path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}
    llm_endpoint = cfg.get("llm_endpoint_name")
    vs = cfg.get("vector_search", {}) or {}
    index_name = vs.get("index_name")
    num_results = int(vs.get("num_results", 5))
    if not llm_endpoint or not index_name:
        raise ValueError("Missing 'llm_endpoint_name' or 'vector_search.index_name' in agent-config-lab.yaml")
    return {
        "llm_endpoint_name": llm_endpoint,
        "vs_index_name": index_name,
        "vs_num_results": num_results,
    }

# Build LangChain agent with LLM and vector search tool
def build_agent(llm_endpoint: str, index_name: str, num_results: int = 5):
    model = ChatDatabricks(endpoint=llm_endpoint, max_tokens=300)
    vs_tool = VectorSearchRetrieverTool(
        name="orion_knowledge_search",
        index_name=index_name,
        description="Search Orion knowledge base for relevant information",
        num_results=num_results,
    )

    system_prompt = (
        "You are the Orion Knowledge Assistant (OKA). Respond in a clear, professional, and factual tone "
        "appropriate for engineers and technical staff. Use only verified information from Orion's internal "
        "documents, and include source references when available. If the answer cannot be found, clearly state "
        "that and suggest related sections or next steps. Do not speculate, make assumptions, or provide "
        "information outside the provided context."
    )
    agent = create_agent(
        model=model,
        tools=[vs_tool],
        system_prompt=system_prompt
    )
    return agent

# Extract last user message from conversation
def _last_user_text(messages: List[Dict[str, Any]]) -> str:
    user_msgs = [m for m in messages if (m.get("role") == "user")]
    return str(user_msgs[-1].get("content", "")) if user_msgs else str(messages[-1].get("content", ""))

# MLflow ResponsesAgent implementation for LangChain agent
class LangChainResponsesAgent(ResponsesAgent):
    def __init__(self):
        cfg = _load_config()
        self._cfg = cfg
        self._agent = build_agent(
            llm_endpoint=cfg["llm_endpoint_name"],
            index_name=cfg["vs_index_name"],
            num_results=cfg["vs_num_results"],
        )

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        msgs = [m.model_dump() for m in request.input]  # [{'role': 'user'|'assistant', 'content': '...'}, ...]
        _ = _last_user_text(msgs) if msgs else ""

        result = self._agent.invoke(
            {"messages": msgs}
        )
        # Extract agent response text
        try:
            text = result["messages"][-1].content
        except Exception:
            text = str(result)

        return ResponsesAgentResponse(
            output=[self.create_text_output_item(text, str(uuid4()))],
            custom_outputs=request.custom_inputs,
        )

# Set the model for mlflow. This is needed when using agent-as-code approach
AGENT = LangChainResponsesAgent()
mlflow.models.set_model(AGENT)

In [0]:
import yaml

def create_config(llm_endpoint_name: str, index_name: str, num_results: int = 3):
    """Create a minimal YAML config for the agent."""
    config = {
        "llm_endpoint_name": llm_endpoint_name,
        "vector_search": {
            "index_name": index_name,
            "num_results": num_results
        }
    }
    return config


# Create config file
llm_endpoint_name = "databricks-gpt-oss-20b"

agent_config = create_config(llm_endpoint_name, vs_index_name)

# Write YAML file (for agent.py to read later)
with open("agent-config-lab.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(agent_config, f, sort_keys=False)

print("✅ Config file written: agent-config-lab.yaml")
print(yaml.safe_dump(agent_config, sort_keys=False))


### B2. Task 4 - Register the Agent Model to Unity Catalog with Alias

Now you will register the agent model to Unity Catalog's Model Registry. This combines logging and registering into a single workflow. You will also add an **alias** to the registered model version.

**Your Task:**

1. Define the model resources (vector search index and serving endpoint).
1. Log the agent model using `mlflow.pyfunc.log_model()` with:
   * Model name: **`"orion_knowledge_assistant_lab"`**
   * Python model: **`"agent-lab.py"`**
   * Code paths: **`["agent-config-lab.yaml"]`**
   * **Model name** to register it to UC model registry.
   * Input example
   * Required pip packages
   * Resources
1. Set an alias for the registered model version.

**Hint:** To learn how to set an alias, refer to the [MLflow Model Registry documentation](https://docs.databricks.com/aws/en/machine-learning/manage-model-lifecycle/#use-model-aliases). 

In [0]:
from mlflow.models.resources import DatabricksVectorSearchIndex, DatabricksServingEndpoint
from importlib.metadata import version as get_version
import mlflow

# Step 1: Define the resources
resources = <FILL_IN>

print("Resources defined:")
for resource in resources:
    print(f"  - {resource}")

# Step 2: Define model configuration
model_name = "orion_knowledge_assistant"
tags_to_register = {
    "model_type": "retrieval_agent",
    "framework": "langchain",
    "use_case": "orion_knowledge_base"
}

input_example = {
    "input": [
        {"role": "user", "content": "What is Orion?"}
    ]
}

# Step 3: Log the model
with mlflow.start_run():
    mlflow.set_tags(tags_to_register)
    
    logged_agent_info = <FILL_IN>
    
    model_uri = logged_agent_info.model_uri
    
print(f"✅ Model logged successfully!")
print(f"Model URI: {model_uri}")

# Step 4: Register the model to Unity Catalog
mlflow.set_registry_uri("databricks-uc")
UC_MODEL_NAME = f"{catalog}.{schema}.orion_knowledge_assistant_lab"

uc_registered_model_info = <FILL_IN>

print(f"✅ Model registered successfully to Unity Catalog!")
print(f"Model Name: {UC_MODEL_NAME}")
print(f"Version: {uc_registered_model_info.version}")

# Step 5: Set an alias for the registered model version
# Refer to the documentation to learn how to set an alias
client = <FILL_IN>
<FILL_IN>

print(f"✅ Alias 'Champion' set for version {uc_registered_model_info.version}")

In [0]:
%skip
from mlflow.models.resources import DatabricksVectorSearchIndex, DatabricksServingEndpoint
from importlib.metadata import version as get_version
import mlflow

# Step 1: Define the resources
resources = [
    DatabricksVectorSearchIndex(index_name=vs_index_name),
    DatabricksServingEndpoint(endpoint_name=llm_endpoint_name)
]

print("Resources defined:")
for resource in resources:
    print(f"  - {resource}")

# Step 2: Define model configuration
model_name = "orion_knowledge_assistant_lab"
UC_MODEL_NAME = f"{catalog}.{schema}.orion_knowledge_assistant_lab"

input_example = {
    "input": [
        {"role": "user", "content": "What is Orion?"}
    ]
}

# Step 3: Log and register the model in one step
mlflow.set_registry_uri("databricks-uc")
with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name=model_name,
        python_model="agent-lab.py",
        code_paths=["agent-config-lab.yaml"],
        input_example=input_example,
        pip_requirements=[
            f"databricks-vectorsearch=={get_version('databricks-vectorsearch')}",
            f"databricks-langchain=={get_version('databricks-langchain')}",
            f"langchain=={get_version('langchain')}",
            f"mlflow=={get_version('mlflow')}",
        ],
        resources=resources,
        registered_model_name=UC_MODEL_NAME
    )
    model_uri = logged_agent_info.model_uri
    model_version = logged_agent_info.registered_model_version

print(f"✅ Model logged and registered successfully!")
print(f"Model URI: {model_uri}")
print(f"Model Name: {UC_MODEL_NAME}")
print(f"Version: {model_version}")

# Step 4: Set an alias for the registered model version
client = mlflow.MlflowClient()
client.set_registered_model_alias(
    name=UC_MODEL_NAME,
    alias="Champion",
    version=model_version
)

print(f"✅ Alias 'Champion' set for version {model_version}")

## C. Testing the Registered Model

In this final section, you will test the registered model to validate that it works correctly. You will load the model from Unity Catalog and make predictions.

### C1. Task 5 - Test the Registered Model

Now that your agent is registered in Unity Catalog, you will test it to verify functionality.

**Your Task:**

1. Use the model URI from the previous step to make a prediction.
1. Use the input example that was logged with the model.
1. Print the agent's response.


In [0]:
# Test the registered model by making a prediction

query = <FILL_IN>

result = <FILL_IN>

print("Agent Response:")
print(result)

In [0]:
%skip
import mlflow

query = {
    "input": [
        {"role": "user", "content": "What are the safety procedures for the Orion?"}
    ]
}

# Make a prediction using the model URI
result = mlflow.models.predict(
    model_uri=model_uri,
    input_data=query,
    env_manager="uv",
)

print("Agent Response:")
print(result)

### C2. Task 6 - Explore the Model Registry UI

With your agent successfully registered in Unity Catalog, you can now explore the Model Registry UI to understand how to manage, monitor, and govern your model.

**Your Tasks:**

- Identify the four main tabs in the Model Registry UI and describe their purpose.
- Locate the **model requirements file** that was logged as an artifact with your model.
- Find the **alias** you set for your registered model version.
- Review the **execution traces** for agent invocations.
- Summarize how the Model Registry UI supports **model lifecycle management** and **governance**.

## D. Summary

You have successfully built, logged, and registered a production-ready retrieval agent using Databricks Mosaic AI.

In this lab, you:

* **Enabled MLflow tracing** for LangChain to monitor agent behavior and execution.
* **Built a retrieval agent** using LangChain with vector search integration.
* **Analyzed agent traces** to identify performance characteristics and understand execution flow.
* **Wrote agent code** to a Python file following the "agent as code" pattern.
* **Registered the agent** to Unity Catalog's Model Registry with proper versioning and aliases.
* **Tested the registered model** to validate functionality.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>